## Paths

In [1]:
from pathlib import Path

import sys

source_path = Path("./").resolve()
sys.path.append(str(source_path))

%load_ext autoreload
%autoreload 2
# from itables import init_notebook_mode, show
# init_notebook_mode(all_interactive=True)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload



## Imports

In [2]:
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

from src.back.data import split_dataset
from src.back.pipeline import *
from src.back.plots import plot_result
from src.back.preprocess import *
from src.back.whatif import *

# DATA INGESTION

## Create DB from csv and metadata
Questa parte non è necessaria, una volta costruito il db definitivo può essere saltata

In [3]:
# logger.info("Converting CSV and Excel files to DuckDB...")
meta_input = load_metadata(meta_dir=CONFIG_META_PATH)
get_data_all(meta_input, db_dir=DB_DIR, data_dir=DATA_DIR)
create_mapping(db_dir=DB_DIR)
anonymize_data(db_dir = DB_DIR)
# reduce_db(db_path = DB_PATH, lst_eco_cod=["ECO_089", "ECO_239", "ECO_044", "ECO_090", "ECO_091", "ECO_132", "ECO_240", "ECO_241", "ECO_043", "ECO_045", "ECO_275"])       
# "ECO_111", "ECO_106", "ECO_112", "ECO_133", "ECO_110", "ECO_107", "ECO_162", "ECO_161", "ECO_109", "ECO_297", 
# reduce_db(db_path = DB_PATH, lst_eco_cod=["ECO_335"]) 
# reduce_db(db_path = DB_PATH, lst_eco_cod=["ECO_111", "ECO_106", "ECO_112"]) 
# reduce_db(db_path = DB_PATH, lst_eco_cod=["ECO_089", "ECO_239", "ECO_044"]) 
# reduce_db(db_path = DB_PATH, lst_eco_cod=["ECO_088", "ECO_237", "ECO_044", "ECO_089", "ECO_090 " ]) 
# reduce_db(db_path = DB_PATH, lst_eco_cod=["ECO_088", "ECO_237", "ECO_234" ]) 
# reduce_db(db_path = DB_PATH, lst_eco_cod=["ECO_223", "ECO_037"]) 
reduce_db(db_path = DB_PATH, lst_eco_cod=["ECO_334"]) 



[    INFO    ] Loading data DRV_ANAG
[    INFO    ] Loaded data shape for DRV_ANAG: (225, 8)
[    INFO    ] Loading data DRV_VAL
[    INFO    ] Loaded data shape for DRV_VAL: (24464, 3)
[    INFO    ] Loading data ECO_ANAG
[    INFO    ] Loaded data shape for ECO_ANAG: (346, 8)
[    INFO    ] Loading data ECO_VAL
[    INFO    ] Loaded data shape for ECO_VAL: (27693, 3)
[    INFO    ] Loading data ECO_DRV_RULES
[    INFO    ] Loaded data shape for ECO_DRV_RULES: (387, 4)
[    INFO    ] Loading data ECO_BUDGET
[    INFO    ] Loaded data shape for ECO_BUDGET: (4191, 3)
[    INFO    ] Database /home/anna/prj/ilabs-bankfcs/bankfcs/src/db/CE/bankfcs_cleaned.db esistente cancellato.
[    INFO    ] Writing table DRV_ANAG to DuckDB
[    INFO    ] Table DRV_ANAG written to DuckDB successfully
[    INFO    ] Writing table DRV_VAL to DuckDB
[    INFO    ] Table DRV_VAL written to DuckDB successfully
[    INFO    ] Writing table ECO_ANAG to DuckDB
[    INFO    ] Table ECO_ANAG written to DuckDB suc

# INPUT

## Load data (D1 + E1)


In [4]:
data_loader = DataLoader(DB_PATH)
logger.info(f"Economics loaded: {data_loader.eco_anag['ECO_COD'].unique()}")
logger.info(f"Drivers loaded: {data_loader.driver_anag['DRV_COD'].unique()}")

[  WARNING   ] Could not load table all_driver_best_model: setting to None.
[  WARNING   ] Could not load table all_driver_selected: setting to None.
[  WARNING   ] Could not load table all_driver_fit: setting to None.
[  WARNING   ] Could not load table all_driver_forecast: setting to None.
[  WARNING   ] Could not load table all_eco_best_model: setting to None.
[  WARNING   ] Could not load table all_eco_fit: setting to None.
[  WARNING   ] Could not load table all_eco_forecast: setting to None.
[  WARNING   ] Could not load table all_forecast_rec: setting to None.
[  WARNING   ] Could not load table all_val_group: setting to None.
[    INFO    ] Cutting 0 drivers due to high missing values
[    INFO    ] Cutting 0 eco due to high missing values
[  WARNING   ] Could not load table all_cause_effect: setting to None.
[    INFO    ] Economics loaded: ['ECO_334']
[    INFO    ] Drivers loaded: ['DRV_038' 'DRV_039' 'DRV_040' 'DRV_103' 'DRV_104' 'DRV_105' 'DRV_106'
 'DRV_107' 'DRV_108' 'DR

In [5]:
check_drv_anag = data_loader.driver_anag
check_drv_df = data_loader.driver_df
check_eco_anag = data_loader.eco_anag
check_eco_df = data_loader.eco_df
check_eco_budget = data_loader.eco_budget


In [0]:

# import pandas as pd
# check_eco_budget = pd.read_csv('C:/dev/ILAB/ilabs-bankfcs/bankfcs/src/data/CE/eco_budget_noaggr.csv', sep=';')
# check_eco_budget['VALUE_FORECAST'] = check_eco_budget.loc[:, 'VALUE_FORECAST'].map(lambda v: v.replace(",", "."))
# check_eco_budget.loc[:, 'VALUE_FORECAST'] = check_eco_budget.loc[:, 'VALUE_FORECAST'].apply(pd.to_numeric, errors="coerce")
# check_eco_budget = check_eco_budget.groupby(['MONTH_FORECAST', 'ECO_COD'], as_index=False)['VALUE_FORECAST'].sum()
# check_eco_budget.to_csv("check_eco_budget.csv", index=False, sep=";")

# TRAIN

## Drivers best model

In [10]:
forecasting_model_drv = ForecastingModel(configuration_path=DRIVERS_MODEL_CONFIG_PATH, freq=config.forecast.freq)
logger.info("Performing driver best model search...")
driver_best_model_search(data_loader, forecasting_model_drv)

[    INFO    ] Successfully created 5 models:
[    INFO    ] • Model: arima_12        | Type: AutoARIMA
[    INFO    ]   └─ Season Length: 12
[    INFO    ] • Model: arima_1         | Type: AutoARIMA
[    INFO    ]   └─ Season Length: 1
[    INFO    ] • Model: mstl_12         | Type: MSTL
[    INFO    ]   └─ Season Length: [12]
[    INFO    ] • Model: mfles_12        | Type: AutoMFLES
[    INFO    ]   └─ Season Length: 12
[    INFO    ] • Model: HistoricAverage_0 | Type: HistoricAverage
[    INFO    ] Performing driver best model search...
[    INFO    ] DRV_038: identified 0 outliers on 121 obs. (0.00%)
[    INFO    ] DRV_039: identified 0 outliers on 133 obs. (0.00%)
[    INFO    ] DRV_040: identified 0 outliers on 133 obs. (0.00%)
[    INFO    ] DRV_103: identified 4 outliers on 133 obs. (3.01%)
[    INFO    ] DRV_104: identified 2 outliers on 133 obs. (1.50%)
[    INFO    ] DRV_105: identified 0 outliers on 133 obs. (0.00%)
[    INFO    ] DRV_106: identified 1 outliers on 133 obs. 

364.55s - thread._ident is None in _get_related_thread!


In [11]:
check_drv_best = data_loader.driver_best_model_df
check_drv_best_stat = data_loader.driver_best_stat_model_df

## Drivers selection

In [12]:

logger.info("Performing driver selection...")
driver_selection(data_loader, config.driver_selection.model_selection, JSON_FILE_PATH)

[    INFO    ] Performing driver selection...


ECO_COD: ECO_334:   0%|          | 0/1 [00:00<?, ?it/s]

[    INFO    ] Removed 0 with bad forecast.
[    INFO    ] ECO_COD ECO_334 has 12 potential drivers.
[    INFO    ] ECO_COD ECO_334 has 121 relevant timestamps.
[    INFO    ] Removed 3 correlated drivers {'DRV_103', 'DRV_104', 'DATE_RIF'}.


ECO_COD: ECO_334: 100%|██████████| 1/1 [00:00<00:00,  4.24it/s]


,ECO_COD,DRV_COD,MODEL,COEFF
0,ECO_334,DRV_039,lasso,0.437433
1,ECO_334,DRV_105,lasso,-1.000000


In [15]:
check_drv_sel= data_loader.driver_selected_df
check_drv_sel = check_drv_sel.merge(data_loader.driver_anag, on="DRV_COD", how="left")

## Drivers forecasting

In [16]:
logger.info("Forecasting drivers using the best model...")
forcast_drivers(data_loader, forecasting_model_drv)

[    INFO    ] Forecasting drivers using the best model...
[    INFO    ] DRV_039: identified 0 outliers on 133 obs. (0.00%)
[    INFO    ] DRV_105: identified 0 outliers on 133 obs. (0.00%)
[    INFO    ] Forecasting series DRV_039 using its best model arima_12
[    INFO    ] Forecasting series DRV_105 using its best model arima_12


In [17]:

check_drv_fore = data_loader.drivers_forecast_df

## Da cancellare, solo per demo

In [0]:
# import numpy as np

# drv_fore = data_loader.drivers_forecast_df
# drv_fore = drv_fore.merge(data_loader.driver_anag, on="DRV_COD", how="left")
# drv_fore_ex = drv_fore[drv_fore["DRV_TYP_0"] == "EXO"].reset_index(drop=True)
# drv_df = data_loader.driver_df
# drv_fore_ex = drv_fore_ex.merge(drv_df, on=["DRV_COD", "DATE_RIF"], how="left")
# drv_fore_ex = drv_fore_ex[drv_fore_ex["VALUE"].notna()].reset_index(drop=True)
# drv_fore_ex["FORECAST_REAL"] = drv_fore_ex["VALUE"]
# drv_fore_ex = drv_fore_ex[["DRV_COD", "DATE_RIF", "FORECAST_REAL"]]
# drv_fore = drv_fore.merge(drv_fore_ex, on=["DRV_COD", "DATE_RIF"], how="left")
# drv_fore["FORECAST"] = drv_fore.apply(lambda x: x["FORECAST_REAL"] if pd.notna(x["FORECAST_REAL"]) else x["FORECAST"], axis=1)

# mask = drv_fore["DRV_TYP_0"] == "ENDO"
# # Esempio: aggiungi rumore normale con media 0 e deviazione standard 0.01 * valore
# drv_fore.loc[mask, "FORECAST"] += np.random.normal(
#     loc=0,
#     scale=0.01 * drv_fore.loc[mask, "FORECAST"].abs()
# )

# drv_fore = drv_fore[["DRV_COD", "DATE_RIF", "FORECAST"]]
# data_loader.drivers_forecast_df = drv_fore.copy()

## Economics best model

In [18]:
logger.info("Cross-validating economics forecasting models...")
forecasting_model_eco = ForecastingModel(configuration_path=ECONOMICS_MODEL_CONFIG_PATH, freq=config.forecast.freq)
# cross_validate_economics_parallel(data_loader, forecasting_model_eco, n_jobs=8)
cross_validate_economics(data_loader, forecasting_model_eco)

[    INFO    ] Cross-validating economics forecasting models...
[    INFO    ] Successfully created 3 models:
[    INFO    ] • Model: arima_12        | Type: AutoARIMA
[    INFO    ]   └─ Season Length: 12
[    INFO    ] • Model: arima_1         | Type: AutoARIMA
[    INFO    ]   └─ Season Length: 1
[    INFO    ] • Model: mfles_12        | Type: AutoMFLES
[    INFO    ]   └─ Season Length: 12


ECO_COD: ECO_334:   0%|          | 0/1 [00:00<?, ?it/s]

[    INFO    ] Eco ECO_334 has selected drivers ['DRV_039' 'DRV_105']
[    INFO    ] ECO_334: identified 1 outliers on 121 obs. (0.83%)


ECO_COD: ECO_334: 100%|██████████| 1/1 [00:38<00:00, 38.37s/it]


In [19]:
check_eco_best = data_loader.eco_best_model_df
check_eco_best_stat = data_loader.eco_best_stat_model_df

## Economics forecasting

In [20]:
logger.info("Forecasting economics using the best model...")
forecast_economics(data_loader, forecasting_model_eco)

[    INFO    ] Forecasting economics using the best model...


ECO_COD: ECO_334:   0%|          | 0/1 [00:00<?, ?it/s]

[    INFO    ] Eco ECO_334 has selected drivers ['DRV_039' 'DRV_105']
[    INFO    ] ECO_334: identified 1 outliers on 121 obs. (0.83%)
[    INFO    ] Forecasting series ECO_334 using its best model arima_12


ECO_COD: ECO_334: 100%|██████████| 1/1 [00:09<00:00,  9.46s/it]


In [21]:
check_eco_fore = data_loader.eco_forecast_df

In [0]:
# data_loader.save_all()

# RECONCILIATION

In [22]:
forecasting_model_rec = ForecastingModel(configuration_path=DRIVERS_MODEL_CONFIG_PATH, freq=config.forecast.freq)
df_rec = fit_reconciliation(data_loader, forecasting_model_rec, aggregation_spec=config.forecast.spec)


[    INFO    ] Successfully created 5 models:
[    INFO    ] • Model: arima_12        | Type: AutoARIMA
[    INFO    ]   └─ Season Length: 12
[    INFO    ] • Model: arima_1         | Type: AutoARIMA
[    INFO    ]   └─ Season Length: 1
[    INFO    ] • Model: mstl_12         | Type: MSTL
[    INFO    ]   └─ Season Length: [12]
[    INFO    ] • Model: mfles_12        | Type: AutoMFLES
[    INFO    ]   └─ Season Length: 12
[    INFO    ] • Model: HistoricAverage_0 | Type: HistoricAverage
[    INFO    ] 20: identified 1 outliers on 121 obs. (0.83%)
[    INFO    ] 20/55: identified 1 outliers on 121 obs. (0.83%)
[    INFO    ] 20/55/_: identified 1 outliers on 121 obs. (0.83%)
[    INFO    ] 20: identified 1 outliers on 121 obs. (0.83%)
[    INFO    ] 20/55: identified 1 outliers on 121 obs. (0.83%)
[    INFO    ] 20/55/_: identified 1 outliers on 121 obs. (0.83%)
[    INFO    ] Forecasting series 20 using its best model arima_1
[    INFO    ] Forecasting series 20/55 using its best model

# WHAT-IF


## Calculate Cause-Effect Matrix

In [0]:
cause_effect_matrix = calculate_cause_effect(data_loader)

# SAVING

In [0]:
data_loader.save_all()

## Propagate shocks

In [0]:
# df_driver_forecast_original = data_loader.drivers_forecast_df

# shock_driver = 'DRV_025'
# shock_month_index = 3
# shock_value = 600

# shock_driver = 'DRV_026'
# shock_month_index = 3
# shock_value = 0.09

# shock_driver = 'DRV_009'
# shock_month_index = 3
# shock_value = 43

# shock_driver = 'DRV_057'
# shock_month_index = 5
# shock_value = 400000

# shock_driver = 'DRV_035'
# shock_month_index = 3
# shock_value = 11



# df_driver_forecast_updated = propagate_shock(
#     data = data_loader,
#     shock_driver = shock_driver,
#     shock_month_index = shock_month_index, 
#     shock_value = shock_value)


In [0]:
# import seaborn as sns
# import matplotlib.pyplot as plt

# check = df_driver_forecast_original.merge(df_driver_forecast_updated, on=['DATE_RIF', 'DRV_COD'], suffixes=('_ORIG', '_WHATIF'))
# check_diff = check[check['FORECAST'] != check['FORECAST_WHATIF']]
# check = check[check['DRV_COD'].isin(check_diff['DRV_COD'].unique())]

# # Assumendo che 'check' abbia colonne: DRV_COD, DATE_RIF, FORECAST, FORECAST_WHATIF
# df_plot = check.melt(
#     id_vars=['DRV_COD', 'DATE_RIF'],
#     value_vars=['FORECAST', 'FORECAST_WHATIF'],
#     var_name='Scenario',
#     value_name='Value'
# )

# # Assumendo che df_plot sia già creato come nel tuo codice precedente
# g = sns.FacetGrid(df_plot, col="DRV_COD", col_wrap=4, sharey=False, height=4)
# g.map_dataframe(sns.lineplot, x="DATE_RIF", y="Value", hue="Scenario")
# # Dopo g.map_dataframe(...)
# for ax, drv_cod in zip(g.axes.flat, g.col_names):
#     if drv_cod == shock_driver:
#         # Ottieni la data corrispondente allo shock
#         shock_date = df_plot[df_plot['DRV_COD'] == shock_driver]['DATE_RIF'].unique()[shock_month_index]
#         ax.scatter(shock_date, shock_value, color='red', s=80, zorder=10)
# g.add_legend()
# g.set_titles(col_template="{col_name}")
# g.set_axis_labels("DATE_RIF", "Valore")
# plt.tight_layout()
# plt.show()

In [0]:
# check_ce_matrix[check_ce_matrix['DRV_COD_CE'] + '_L1' == check_ce_matrix['DRV_COD']].reset_index(drop=True)  